In [ ]:
!pip -q install transformers peft
import os, torch, numpy as np, random
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/thesis_ckpts')
DATA  = Path('/content/polyp')
DATA.mkdir(parents=True, exist_ok=True)

# Zenodo 5579392 — PraNet package (train + 5 test set)
!wget -q -O /content/polyp/TrainDataset.zip "https://zenodo.org/records/5579392/files/TrainDataset.zip?download=1"
!wget -q -O /content/polyp/TestDataset.zip  "https://zenodo.org/records/5579392/files/TestDataset.zip?download=1"
!cd /content/polyp && unzip -q TrainDataset.zip && unzip -q TestDataset.zip
!ls /content/polyp/TestDataset

In [ ]:
!ls -la /content/polyp
!file /content/polyp/TrainDataset.zip
!head -c 300 /content/polyp/TrainDataset.zip

In [ ]:
import requests
r = requests.get('https://zenodo.org/api/records/5579392', timeout=30)
print('status:', r.status_code)
print(r.text[:500])

In [ ]:
d = r.json()
print(d['metadata']['title'])
for f in d.get('files', []):
    print(f['key'], round(f['size']/1e6,1), 'MB')
    print('   ', f['links']['self'])

In [ ]:
!wget -c -O /content/polyp/PolypDataset.zip \
  "https://zenodo.org/api/records/5579392/files/PolypDataset.zip/content"
!file /content/polyp/PolypDataset.zip && du -h /content/polyp/PolypDataset.zip

In [ ]:
!ls /content/polyp

In [ ]:
!cd /content/polyp && unzip -q PolypDataset.zip && rm PolypDataset.zip
!find /content/polyp -maxdepth 3 -type d | head -30

In [ ]:
from pathlib import Path
P = Path('/content/polyp')

for d in ['OLDTRimage','OLDTRimageMask','NewTRimage','NewTRmask','NewTRimageDA','NewTRmaskDA']:
    p = P/d
    n = len(list(p.iterdir())) if p.exists() else -1
    ex = sorted(x.name for x in p.iterdir())[:3] if p.exists() else []
    print(f'{d:16s} {n:6d}  {ex}')

T = P/'TestDataset/TestDataset'
for d in sorted(x.name for x in T.iterdir()):
    ni = len(list((T/d/'images').iterdir()))
    nm = len(list((T/d/'masks').iterdir()))
    print(f'{d:20s} images={ni:4d} masks={nm:4d}')

In [ ]:
#cell2
import random
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

IMG_SIZE = 224
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)

class PolypDataset(Dataset):
    """
    Two modes:
      nested  : root/images + root/masks           (test sets)
      flat    : img_dir + msk_dir given separately (NewTRimage / NewTRmask)
    File names match on both sides (1.png -> 1.png).
    """
    def __init__(self, img_dir, msk_dir=None, train=False, frac=1.0, seed=42):
        self.img_dir = Path(img_dir)
        self.msk_dir = Path(msk_dir) if msk_dir else self.img_dir.parent/'masks'
        if msk_dir is None:                      # nested mode
            self.img_dir = Path(img_dir)/'images'
            self.msk_dir = Path(img_dir)/'masks'
        names = sorted(p.name for p in self.img_dir.iterdir()
                       if p.suffix.lower() in ('.png','.jpg','.jpeg','.tif'))
        if frac < 1.0:                           # data-size sweep
            rng = random.Random(seed); shuf = names[:]; rng.shuffle(shuf)
            names = sorted(shuf[:max(1, int(len(shuf)*frac))])
        self.names, self.train = names, train
        self.norm = T.Normalize(MEAN, STD)
        self.jitter = T.ColorJitter(0.2,0.2,0.2,0.05)

    def __len__(self): return len(self.names)

    def _mask_path(self, name):
        stem = Path(name).stem
        for ext in ('.png','.jpg','.jpeg','.tif'):
            p = self.msk_dir/f'{stem}{ext}'
            if p.exists(): return p
        raise FileNotFoundError(f'no mask: {stem} @ {self.msk_dir}')

    def __getitem__(self, i):
        name = self.names[i]
        img = Image.open(self.img_dir/name).convert('RGB')
        msk = Image.open(self._mask_path(name)).convert('L')
        img = TF.resize(img, (IMG_SIZE,IMG_SIZE), TF.InterpolationMode.BILINEAR)
        msk = TF.resize(msk, (IMG_SIZE,IMG_SIZE), TF.InterpolationMode.NEAREST)

        if self.train:   # train-time augmentation, identical across the three methods
            if random.random() < 0.5: img, msk = TF.hflip(img), TF.hflip(msk)
            if random.random() < 0.5: img, msk = TF.vflip(img), TF.vflip(msk)
            k = random.choice([0,90,180,270])
            if k: img, msk = TF.rotate(img,k), TF.rotate(msk,k)
            img = self.jitter(img)

        img = self.norm(TF.to_tensor(img))
        msk = (TF.to_tensor(msk) > 0.5).float()
        return img, msk, name

In [ ]:
#cell 3
P = Path('/content/polyp')
TR_IMG, TR_MSK = P/'NewTRimage', P/'NewTRmask'
T5 = P/'TestDataset/TestDataset'

full_train = PolypDataset(TR_IMG, TR_MSK, train=True)   # 1450
assert len(full_train) == 1450

# val split: all three methods see the SAME val set (early-stopping symmetry)
g = torch.Generator().manual_seed(42)
idx = torch.randperm(len(full_train), generator=g).tolist()
n_val = int(0.1*len(full_train))                        # 145
val_idx, tr_idx = idx[:n_val], idx[n_val:]              # 145 / 1305

# separate instance for val with augmentation OFF
val_base = PolypDataset(TR_IMG, TR_MSK, train=False)
train_ds = torch.utils.data.Subset(full_train, tr_idx)
val_ds   = torch.utils.data.Subset(val_base,  val_idx)
print('train', len(train_ds), '| val', len(val_ds))

test_sets = {
    'Kvasir'      : PolypDataset(T5/'Kvasir'),             # in-domain
    'CVC-ClinicDB': PolypDataset(T5/'CVC-ClinicDB'),       # in-domain
    'CVC-T'       : PolypDataset(T5/'CVC-300'),            # OOD
    'CVC-ColonDB' : PolypDataset(T5/'CVC-ColonDB'),        # OOD
    'ETIS'        : PolypDataset(T5/'ETIS-LaribPolypDB'),  # OOD
}
test_loaders = {k: DataLoader(v, batch_size=16, shuffle=False, num_workers=2)
                for k,v in test_sets.items()}
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)

x,y,n = next(iter(train_loader))
print(x.shape, y.shape, y.unique(), n[:2])

In [ ]:
#cell 3b
import hashlib
def hashes(ds):
    h = set()
    for n in ds.names:
        b = (ds.img_dir/n).read_bytes()
        h.add(hashlib.md5(b).hexdigest())
    return h

tr_h = hashes(PolypDataset(TR_IMG, TR_MSK))
for k, ds in test_sets.items():
    ov = len(tr_h & hashes(ds))
    print(f'{k:14s} overlap={ov} / {len(ds)}')

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModel

dev = 'cuda'
enc = AutoModel.from_pretrained('facebook/dinov2-small').to(dev).eval()

@torch.no_grad()
def feats(loader):
    out = []
    for x, _, _ in loader:
        h = enc(pixel_values=x.to(dev)).last_hidden_state   # (B, 1+N, 384)
        cls, patch = h[:, 0], h[:, 1:].mean(1)
        f = torch.cat([cls, patch], dim=1)                  # (B, 768)
        out.append(F.normalize(f, dim=1).cpu())             # L2-norm
    return torch.cat(out)

def mmd2_linear_unbiased(X, Y):
    X, Y = X.double(), Y.double()
    m, n = len(X), len(Y)
    Kxx = X @ X.T; Kyy = Y @ Y.T; Kxy = X @ Y.T
    t1 = (Kxx.sum() - Kxx.diag().sum()) / (m*(m-1))
    t2 = (Kyy.sum() - Kyy.diag().sum()) / (n*(n-1))
    t3 = 2 * Kxy.mean()
    return (t1 + t2 - t3).item()

# train (augmentation OFF - distance measurement, not training)
train_eval_loader = DataLoader(PolypDataset(TR_IMG, TR_MSK, train=False),
                               batch_size=32, shuffle=False, num_workers=2)
Ftr = feats(train_eval_loader)
print('train feats', Ftr.shape)

# sanity: split train in two, distance to itself should be ~0 (calibration)
half = len(Ftr)//2
print(f"{'train-self':14s} MMD2 = {mmd2_linear_unbiased(Ftr[:half], Ftr[half:]):.4f}")

for k, ld in test_loaders.items():
    d = mmd2_linear_unbiased(Ftr, feats(ld))
    print(f'{k:14s} MMD2 = {d:.4f}  (n={len(test_sets[k])})')

In [ ]:
# 1) Correct self-MMD: split at random
perm = torch.randperm(len(Ftr), generator=torch.Generator().manual_seed(0))
A, B = Ftr[perm[:725]], Ftr[perm[725:]]
print(f"{'train-self(shuf)':20s} MMD2 = {mmd2_linear_unbiased(A,B):.4f}")

# 2) Split train by source (from the filename format
kv_idx  = [i for i,n in enumerate(train_eval_loader.dataset.names) if not n[:-4].isdigit()]
cvc_idx = [i for i,n in enumerate(train_eval_loader.dataset.names) if n[:-4].isdigit()]
print('train Kvasir:', len(kv_idx), '| train ClinicDB:', len(cvc_idx))
Fkv, Fcvc = Ftr[kv_idx], Ftr[cvc_idx]
print(f"{'Kvasir<->ClinicDB':20s} MMD2 = {mmd2_linear_unbiased(Fkv,Fcvc):.4f}")

# 3) Measure each test set against its OWN source subset
for k, ld in test_loaders.items():
    Fte = feats(ld)
    print(f'{k:14s} vs-full={mmd2_linear_unbiased(Ftr,Fte):.4f} '
          f'vs-Kvasir={mmd2_linear_unbiased(Fkv,Fte):.4f} '
          f'vs-ClinicDB={mmd2_linear_unbiased(Fcvc,Fte):.4f}')

In [ ]:
# Reduce all test sets to the same n (60) and measure again
import numpy as np
g2 = torch.Generator().manual_seed(1)
for k, ld in test_loaders.items():
    Fte = feats(ld)
    vals = []
    for s in range(5):
        gi = torch.Generator().manual_seed(s)
        sub = Fte[torch.randperm(len(Fte), generator=gi)[:60]]
        vals.append(mmd2_linear_unbiased(Ftr, sub))
    print(f'{k:14s} MMD2@n=60 = {np.mean(vals):.4f} +/- {np.std(vals):.4f}')

In [ ]:
# check that the data is in place
!ls /content/polyp

In [ ]:
# If the Colab session dropped entirely (/content/polyp empty), just repeat the download:!mkdir -p /content/polyp
!wget -c -O /content/polyp/PolypDataset.zip \
  "https://zenodo.org/api/records/5579392/files/PolypDataset.zip/content"
!cd /content/polyp && unzip -q PolypDataset.zip && rm PolypDataset.zip

Start

In [ ]:
# cell-1 setup
!pip install -q transformers peft albumentations
!pip uninstall -y -q torchao        # version conflict with peft
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/thesis_ckpts'
os.makedirs(CKPT, exist_ok=True)
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import torch
for tag in ['resunet_polyp_lr0.003_e300_s42', 'plainunet_polyp_lr0.001_e300_s42']:
    sd = torch.load(f'{CKPT}/{tag}.pth', map_location='cpu')
    if isinstance(sd, dict) and 'state_dict' in sd: sd = sd['state_dict']
    n = sum(v.numel() for v in sd.values())
    widths = sorted({v.shape[0] for v in sd.values() if hasattr(v,'dim') and v.dim() == 4})
    print(f'{tag}\n   parametre: {n:,}\n   kanallar: {widths}\n')

In [ ]:
!pip install -q "transformers==4.57.6" peft albumentations

In [ ]:
#cell 2
import random, numpy as np, math, os
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, PeftModel

IMG_SIZE, PATCH_SIZE = 224, 14
BATCH_SIZE, VAL_SPLIT, SEED = 16, 0.1, 42
DEVICE = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

In [ ]:
#cell 3 - dataset + 5 test set
P  = Path('/content/polyp')
T5 = P/'TestDataset/TestDataset'

class PolypDataset(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False):
        self.image_paths, self.mask_paths = image_paths, mask_paths
        aug = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)] if augment else []
        self.transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *aug,
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, i):
        image = np.array(Image.open(self.image_paths[i]).convert('RGB'))
        mask  = np.array(Image.open(self.mask_paths[i]).convert('L'))
        t = self.transform(image=image, mask=mask)
        return t['image'].float(), (t['mask'].float()/255.0 > 0.5).float().unsqueeze(0)

img_files  = sorted((P/'NewTRimage').glob('*.png'))
mask_files = sorted((P/'NewTRmask').glob('*.png'))
assert len(img_files) == len(mask_files) == 1450
assert all(a.stem == b.stem for a,b in zip(img_files, mask_files)), "name shift"

idx = list(range(1450)); random.Random(SEED).shuffle(idx)
split = int(1450*(1-VAL_SPLIT)); tr, va = idx[:split], idx[split:]

train_ds_aug = PolypDataset([img_files[i] for i in tr], [mask_files[i] for i in tr], augment=True)
val_ds       = PolypDataset([img_files[i] for i in va], [mask_files[i] for i in va], augment=False)
train_loader = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,       batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

def _pair(d):
    im = sorted((T5/d/'images').glob('*'))
    mk = sorted((T5/d/'masks').glob('*'))
    assert len(im)==len(mk) and all(a.stem==b.stem for a,b in zip(im,mk)), f'{d} shift'
    return im, mk

clean_sets = {
    'Kvasir'      : _pair('Kvasir'),
    'CVC-ClinicDB': _pair('CVC-ClinicDB'),
    'CVC-ColonDB' : _pair('CVC-ColonDB'),
    'ETIS'        : _pair('ETIS-LaribPolypDB'),
    'CVC-T'       : _pair('CVC-300'),
}
print('train', len(tr), '| val', len(va))
for k,(i,m) in clean_sets.items(): print(f'  {k:14s} {len(i)}')

In [ ]:
# cell 4 - model classes + loss + metric
class LinearDecoder(nn.Module):
    def __init__(self, embed_dim=384):
        super().__init__()
        self.grid = IMG_SIZE // PATCH_SIZE
        self.proj = nn.Conv2d(embed_dim, 1, 1)
    def forward(self, tokens):
        B, N, D = tokens.shape
        x = tokens.permute(0,2,1).reshape(B, D, self.grid, self.grid)
        x = self.proj(x)
        return F.interpolate(x, size=IMG_SIZE, mode='bilinear', align_corners=False)

class DINOv2SegFrozen(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for p in self.encoder.parameters(): p.requires_grad = False
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return self.decoder(out.last_hidden_state[:,1:,:])

class DINOv2SegLoRA(nn.Module):
    def __init__(self, rank=8):
        super().__init__()
        enc = AutoModel.from_pretrained('facebook/dinov2-small')
        cfg = LoraConfig(r=rank, lora_alpha=rank*2, target_modules=['query','value'],
                         lora_dropout=0.1, bias='none')
        self.encoder = get_peft_model(enc, cfg)
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return self.decoder(out.last_hidden_state[:,1:,:])

class DINOv2SegFull(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for p in self.encoder.parameters(): p.requires_grad = True
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return self.decoder(out.last_hidden_state[:,1:,:])

def dice_bce(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    num = 2*(p*target).sum(dim=(1,2,3)) + 1e-6
    den = p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + 1e-6
    return bce + (1 - (num/den).mean())

@torch.no_grad()
def metrics_per_image(m, loader):
    """per-image DSC and IoU vectors (NOT batch-weighted)."""
    m.eval(); D, I = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast('cuda'):
            p = (torch.sigmoid(m(x)) > 0.5).float()
        inter = (p*y).sum(dim=(1,2,3))
        psum, ysum = p.sum(dim=(1,2,3)), y.sum(dim=(1,2,3))
        D.append(((2*inter+1e-6)/(psum+ysum+1e-6)).cpu())
        I.append(((inter+1e-6)/(psum+ysum-inter+1e-6)).cpu())
    return torch.cat(D).numpy(), torch.cat(I).numpy()

@torch.no_grad()
def eval_vec(m, imgs, masks):
    ds = PolypDataset(imgs, masks, augment=False)
    ld = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)
    return metrics_per_image(m, ld)
print("model classes ready")

In [ ]:
#cell 5 - llrd + scheduler + trainer
def make_warmup_cosine(opt, warmup_ep, total_ep, min_factor=0.01):
    def fn(ep):
        if ep < warmup_ep: return (ep+1)/max(1,warmup_ep)
        prog = (ep-warmup_ep)/max(1,total_ep-warmup_ep)
        return min_factor + (1-min_factor)*0.5*(1+math.cos(math.pi*prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def build_llrd_params(model_full, base_lr_enc, lr_dec, decay=0.75, wd=1e-4):
    enc = model_full.encoder; layers = enc.encoder.layer; n = len(layers)
    groups = [{'params':[p for p in enc.embeddings.parameters() if p.requires_grad],
               'lr': base_lr_enc*(decay**n), 'weight_decay': wd}]
    for i, layer in enumerate(layers):
        groups.append({'params':[p for p in layer.parameters() if p.requires_grad],
                       'lr': base_lr_enc*(decay**(n-1-i)), 'weight_decay': wd})
    groups.append({'params':[p for p in model_full.decoder.parameters() if p.requires_grad],
                   'lr': lr_dec, 'weight_decay': wd})
    return groups

def train_model(m, opt, tag, total_ep=20, warmup_ep=2, patience=5):
    sch = make_warmup_cosine(opt, warmup_ep, total_ep)
    scaler = torch.amp.GradScaler('cuda')
    best, bad = -1, 0
    for ep in range(total_ep):
        m.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = dice_bce(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        vd, _ = metrics_per_image(m, val_loader)
        v = vd.mean()
        flag = ''
        if v > best:
            best, bad = v, 0
            torch.save(m.state_dict(), f'{CKPT}/{tag}.pth'); flag = ' *'
        else:
            bad += 1
        print(f"  ep {ep+1:2d}/{total_ep}  val DSC {v:.4f}{flag}", flush=True)
        if bad >= patience:
            print(f"  early stop (ep {ep+1})"); break
    m.load_state_dict(torch.load(f'{CKPT}/{tag}.pth'))
    return best
print("trainer ready")

def train_or_load(m, opt, tag, **kw):
     """Load the checkpoint if it exists, otherwise train. Avoids retraining after an interruption."""
    p = f'{CKPT}/{tag}.pth'
    if os.path.exists(p):
        m.load_state_dict(torch.load(p))
        vd, _ = metrics_per_image(m, val_loader)
        print(f"  SKIP {tag} (already there) — val DSC {vd.mean():.4f}")
        return vd.mean()
    return train_model(m, opt, tag, **kw)

In [ ]:
# cell 6 - frozen(linear probing)
print("="*50, "\nFROZEN\n", "="*50)
m_fr = DINOv2SegFrozen().to(DEVICE)
opt_fr = torch.optim.AdamW(m_fr.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
b_fr   = train_or_load(m_fr,   opt_fr,   'frozen_polyp_s42', total_ep=20, warmup_ep=2, patience=5)
print(f"frozen val DSC: {b_fr:.4f}")

In [ ]:
x, y = next(iter(train_loader))
print("step/epoch:", len(train_loader))     # 82
print("mask unique:", y.unique())           # [0., 1.]
print("lesion rate:", y.mean().item())     # ~0.1-0.2

In [ ]:
# cell 7 - lora r=8
print("="*50, "\nLoRA r=8\n", "="*50)
m_lr = DINOv2SegLoRA(rank=8).to(DEVICE)
opt_lr = torch.optim.AdamW([p for p in m_lr.parameters() if p.requires_grad],
                           lr=1e-4, weight_decay=1e-4)
b_lr   = train_or_load(m_lr,   opt_lr,   'lora_polyp_s42',   total_ep=20, warmup_ep=2, patience=5)
print(f"LoRA val DSC: {b_lr:.4f}")

In [ ]:
#cell8 - FFT + LLRD
print("="*50, "\nFULL FT (LLRD 0.75)\n", "="*50)
m_full = DINOv2SegFull().to(DEVICE)
opt_full = torch.optim.AdamW(build_llrd_params(m_full, base_lr_enc=5e-6, lr_dec=1e-4, decay=0.75))
b_full = train_or_load(m_full, opt_full, 'full_polyp_s42',   total_ep=20, warmup_ep=3, patience=5)
print(f"full FT val DSC: {b_full:.4f}")

In [ ]:
#cell 9 - pilot result: 5 set x 3 methods
models = {'frozen': m_fr, 'LoRA': m_lr, 'full FT': m_full}
MMD = {'Kvasir':0.010, 'CVC-ClinicDB':0.024, 'CVC-ColonDB':0.113, 'ETIS':0.124, 'CVC-T':0.200}
order = sorted(clean_sets, key=lambda d: MMD[d])   # MMD'ye gore sirala

print(f"\n{'set':14s} {'MMD2':>6s} {'frozen':>8s} {'LoRA':>8s} {'fullFT':>8s} {'D(LoRA-full)':>13s}")
print("-"*62)
pilot = {}
for d in order:
    im, mk = clean_sets[d]
    v = {k: eval_vec(m, im, mk)[0] for k,m in models.items()}   # [0] = DSC
    pilot[d] = v
    delta = v['LoRA'].mean() - v['full FT'].mean()
    print(f"{d:14s} {MMD[d]:6.3f} {v['frozen'].mean():8.4f} {v['LoRA'].mean():8.4f} "
          f"{v['full FT'].mean():8.4f} {delta:+13.4f}")
np.save(f'{CKPT}/pilot_polyp_s42.npy', pilot, allow_pickle=True)

In [ ]:
#Cell 10 — LoRAx class (rank + DoRA + target_modules parametric)
class DINOv2SegLoRAx(nn.Module):
    """Single class: rank sweep + DoRA + target-module all from here."""
    def __init__(self, rank=8, use_dora=False, target_modules=('query','value')):
        super().__init__()
        enc = AutoModel.from_pretrained('facebook/dinov2-small')
        cfg = LoraConfig(r=rank, lora_alpha=rank*2,
                         target_modules=list(target_modules),
                         lora_dropout=0.1, bias='none', use_dora=use_dora)
        self.encoder = get_peft_model(enc, cfg)
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return self.decoder(out.last_hidden_state[:,1:,:])

# parameter counts - to see capacity axis
for r in [4,8,16,32]:
    m = DINOv2SegLoRAx(rank=r)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"LoRA r={r:2d}: {n:,} trainable")
    del m
mf = DINOv2SegFull()
print(f"full FT : {sum(p.numel() for p in mf.parameters() if p.requires_grad):,} trainable")
del mf

In [ ]:
#cell 11 - rank sweep
RANKS = [4, 8, 16, 32]
SWEEP_SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

sweep_models = {}
for r in RANKS:
    tag = f'lora_r{r}_polyp_s42'
    set_seed(SWEEP_SEED)
    print("="*50, f"\nLoRA r={r}\n", "="*50)
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    b = train_or_load(m, opt, tag, total_ep=20, warmup_ep=2, patience=5)
    sweep_models[r] = m
    print(f"  r={r} val DSC: {b:.4f}")

In [ ]:
# cell 12 - sweep eval - vs FFT
MMD = {'Kvasir':0.010, 'CVC-ClinicDB':0.024, 'CVC-ColonDB':0.113, 'ETIS':0.124, 'CVC-T':0.200}
order = ['Kvasir','CVC-ClinicDB','CVC-ColonDB','ETIS','CVC-T']

# full FT reference (m_full should be loaded from Cell 8; if not, load it from the checkpoint)
if 'm_full' not in dir():
    m_full = DINOv2SegFull().to(DEVICE)
    m_full.load_state_dict(torch.load(f'{CKPT}/full_polyp_s42.pth')); m_full.eval()

full_dsc = {d: eval_vec(m_full, *clean_sets[d])[0].mean() for d in order}

print(f"{'set':14s} {'MMD2':>6s} {'fullFT':>7s} " + " ".join(f'{"r="+str(r):>7s}' for r in RANKS))
print("-"*70)
rank_dsc = {}
for d in order:
    row = {r: eval_vec(sweep_models[r], *clean_sets[d])[0].mean() for r in RANKS}
    rank_dsc[d] = row
    print(f"{d:14s} {MMD[d]:6.3f} {full_dsc[d]:7.4f} " +
          " ".join(f"{row[r]:7.4f}" for r in RANKS))

print(f"\nD(LoRA - fullFT), positive = LoRA wins")
print(f"{'set':14s} " + " ".join(f'{"r="+str(r):>8s}' for r in RANKS))
print("-"*55)
for d in order:
    print(f"{d:14s} " + " ".join(f"{rank_dsc[d][r]-full_dsc[d]:+8.4f}" for r in RANKS))

np.save(f'{CKPT}/rank_sweep_polyp.npy',
        {'rank_dsc':rank_dsc,'full_dsc':full_dsc}, allow_pickle=True)

In [ ]:
#cell 13 - multi seed
SEEDS = [42, 123, 2024]
BEST_RANK = 32

ms_models = {}
for method in ['lora', 'full']:
    for s in SEEDS:
        tag = f'{method}_r{BEST_RANK}_polyp_s{s}' if method=='lora' else f'full_polyp_s{s}'
        set_seed(s)
        print("="*50, f"\n{method} | seed {s}\n", "="*50)
        if method == 'lora':
            m = DINOv2SegLoRAx(rank=BEST_RANK).to(DEVICE)
            opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                                    lr=1e-4, weight_decay=1e-4)
            b = train_or_load(m, opt, tag, total_ep=20, warmup_ep=2, patience=5)
        else:
            m = DINOv2SegFull().to(DEVICE)
            opt = torch.optim.AdamW(build_llrd_params(m, 5e-6, 1e-4, decay=0.75))
            b = train_or_load(m, opt, tag, total_ep=20, warmup_ep=3, patience=5)
        ms_models[(method, s)] = m
        print(f"  {method} s{s} val DSC: {b:.4f}")

In [ ]:
from scipy.stats import wilcoxon

order = ['Kvasir','CVC-ClinicDB','CVC-ColonDB','ETIS','CVC-T']
MMD = {'Kvasir':0.010,'CVC-ClinicDB':0.024,'CVC-ColonDB':0.113,'ETIS':0.124,'CVC-T':0.200}

# per-image DSC vector AVERAGED over seeds (not single-seed)
mean_vec = {'lora':{}, 'full':{}}
for method in ['lora','full']:
    for d in order:
        vecs = [eval_vec(ms_models[(method,s)], *clean_sets[d])[0] for s in SEEDS]
        mean_vec[method][d] = np.mean(vecs, axis=0)

print(f"{'set':14s} {'MMD2':>6s} {'LoRA r32':>9s} {'fullFT':>8s} {'D':>8s} {'p':>9s}  winner")
print("-"*68)
for d in order:
    a, b = mean_vec['lora'][d], mean_vec['full'][d]
    stat, p = wilcoxon(a, b)
    delta = a.mean() - b.mean()
    if p < 0.05:  win = 'LoRA' if delta > 0 else 'full FT'
    else:         win = 'tie'
    print(f"{d:14s} {MMD[d]:6.3f} {a.mean():9.4f} {b.mean():8.4f} {delta:+8.4f} {p:9.4f}  {win}")

np.save(f'{CKPT}/multiseed_polyp.npy', mean_vec, allow_pickle=True)

In [ ]:
!mkdir -p /content/drive/MyDrive/polyp_data
!cp -r /content/polyp/NewTRimage /content/polyp/NewTRmask \
       /content/polyp/TestDataset /content/drive/MyDrive/polyp_data/
!du -sh /content/drive/MyDrive/polyp_data

In [ ]:
!ls /content/drive/MyDrive/thesis_ckpts | grep full

In [ ]:
# cell A
import torch, numpy as np
from transformers import AutoModel
import matplotlib.pyplot as plt

@torch.no_grad()
def delta_w_spectra(ckpt_path, n_blocks=12):
    """full FT weights - pretrained weights, SVD of Q/K/V/O for each block."""
    pre = AutoModel.from_pretrained('facebook/dinov2-small')
    pre_sd = {k: v for k, v in pre.state_dict().items()}

    sd = torch.load(ckpt_path, map_location='cpu')
    # checkpoint is saved with the 'encoder.' prefix (DINOv2SegFull wrapper)
    ft_sd = {k[len('encoder.'):]: v for k, v in sd.items() if k.startswith('encoder.')}

    out = {}
    for b in range(n_blocks):
        for name in ['query', 'key', 'value']:
            key = f'encoder.layer.{b}.attention.attention.{name}.weight'
            if key not in ft_sd or key not in pre_sd:
                continue
            dW = (ft_sd[key].float() - pre_sd[key].float())
            s = torch.linalg.svdvals(dW).numpy()
            out[(b, name)] = s
    del pre
    return out

def spectral_summary(s, thresholds=(0.90, 0.95, 0.99)):
    """Number of singular values capturing X% of the energy + effective rank (entropy)."""
    e = s**2
    c = np.cumsum(e) / e.sum()
    res = {f'r@{int(t*100)}': int(np.searchsorted(c, t) + 1) for t in thresholds}
    p = e / e.sum()
    res['eff_rank'] = float(np.exp(-(p * np.log(p + 1e-12)).sum()))  # entropy based
    res['s1_over_s10'] = float(s[0] / s[9]) if len(s) > 9 else np.nan
    return res

In [ ]:
# cell b
SEEDS = [42, 123, 2024]
DOMAINS = {
    'skin' : [f'{CKPT}/full_s{s}.pth'       for s in SEEDS],
    'polyp': [f'{CKPT}/full_polyp_s{s}.pth' for s in SEEDS],
}

spectra = {}
for dom, paths in DOMAINS.items():
    per_seed = []
    for p in paths:
        if not os.path.exists(p):
            print(f'MISSING: {p}'); continue
        per_seed.append(delta_w_spectra(p))
    spectra[dom] = per_seed
    print(f'{dom}: {len(per_seed)} seeds loaded')

# block average, seed average
print(f"\n{'domain':8s} {'r@90':>6s} {'r@95':>6s} {'r@99':>6s} {'eff_rank':>9s} {'s1/s10':>8s}")
print('-'*50)
summary = {}
for dom, per_seed in spectra.items():
    rows = []
    for sp in per_seed:
        for k, s in sp.items():
            rows.append(spectral_summary(s))
    agg = {k: np.mean([r[k] for r in rows]) for k in rows[0]}
    summary[dom] = agg
    print(f"{dom:8s} {agg['r@90']:6.1f} {agg['r@95']:6.1f} {agg['r@99']:6.1f} "
          f"{agg['eff_rank']:9.1f} {agg['s1_over_s10']:8.2f}")

In [ ]:
# cell c
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for dom, per_seed in spectra.items():
    curves = []
    for sp in per_seed:
        for k, s in sp.items():
            curves.append(s / s[0])          # s1'e normalize
    m = np.mean(np.stack(curves), axis=0)
    sd = np.std(np.stack(curves), axis=0)
    x = np.arange(1, len(m)+1)
    ax[0].plot(x, m, label=dom, lw=2)
    ax[0].fill_between(x, m-sd, m+sd, alpha=0.2)

    e = np.mean(np.stack([c**2/np.sum(c**2) for c in curves]), axis=0)
    ax[1].plot(x, np.cumsum(e), label=dom, lw=2)

ax[0].set(xlabel='singular value index', ylabel='normalised magnitude',
          title='dW spectrum (mean over blocks x seeds)', yscale='log', xlim=(1,64))
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].axhline(0.9, ls='--', c='gray', lw=1); ax[1].axhline(0.95, ls='--', c='gray', lw=1)
ax[1].set(xlabel='rank', ylabel='cumulative energy',
          title='cumulative spectral energy', xlim=(1,64))
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{CKPT}/dW_spectra.png', dpi=150); plt.show()

In [ ]:
# cell e
import torch, numpy as np, copy
from transformers import AutoModel

@torch.no_grad()
def build_truncated_full(ft_ckpt, ranks=(4,8,16,32), n_blocks=12):
    """
    W_pre + truncate_r(W_full - W_pre) for EVERY 2D weight the full FT changed.
    1D params (LayerNorm, bias) are kept at their full-FT value (no rank to truncate).
    This gives the rank-r approximation of the COMPLETE full-FT solution,
    which is the fair comparison against LoRA at rank r.
    """
    pre = AutoModel.from_pretrained('facebook/dinov2-small')
    pre_sd = pre.state_dict()
    sd = torch.load(ft_ckpt, map_location='cpu')
    ft_enc = {k[len('encoder.'):]: v for k, v in sd.items() if k.startswith('encoder.')}
    ft_dec = {k: v for k, v in sd.items() if k.startswith('decoder.')}

    # find every encoder weight that full FT actually moved
    changed = {}
    for k in pre_sd:
        if k in ft_enc and not torch.equal(pre_sd[k], ft_enc[k]):
            changed[k] = (pre_sd[k].float(), ft_enc[k].float())

    n_2d = sum(1 for _, (p, _) in changed.items() if p.ndim == 2)
    print(f"changed encoder tensors: {len(changed)} total, {n_2d} are 2D (truncatable)")

    models = {}
    for r in ranks:
        new_enc = copy.deepcopy(pre_sd)
        for k, (Wp, Wf) in changed.items():
            dW = Wf - Wp
            if Wp.ndim == 2 and min(Wp.shape) > r:
                U, S, Vh = torch.linalg.svd(dW, full_matrices=False)
                dW = (U[:, :r] * S[:r]) @ Vh[:r, :]        # rank-r truncation
            new_enc[k] = Wp + dW                           # 1D or small: keep full update
        m = DINOv2SegFull().to(DEVICE)
        m.encoder.load_state_dict(new_enc)
        m.decoder.load_state_dict({k[len('decoder.'):]: v for k, v in ft_dec.items()})
        m.eval()
        models[r] = m
    del pre
    return models

trunc = build_truncated_full(f'{CKPT}/full_polyp_s42.pth', ranks=(4, 8, 16, 32))
print("truncated models ready:", list(trunc.keys()))

In [ ]:
# cell d
order = ['Kvasir', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS', 'CVC-T']

def load_lora(r):
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    m.load_state_dict(torch.load(f'{CKPT}/lora_r{r}_polyp_s42.pth')); m.eval(); return m

# full FT reference (untruncated)
m_full_ref = DINOv2SegFull().to(DEVICE)
m_full_ref.load_state_dict(torch.load(f'{CKPT}/full_polyp_s42.pth')); m_full_ref.eval()

print(f"{'set':14s} {'full':>7s} | " + " ".join(f'{"tr"+str(r):>7s} {"lo"+str(r):>7s}' for r in (4, 32)))
print('-' * 70)
for d in order:
    im, mk = clean_sets[d]
    full = eval_vec(m_full_ref, im, mk)[0].mean()
    row = []
    for r in (4, 32):
        t = eval_vec(trunc[r], im, mk)[0].mean()
        l = eval_vec(load_lora(r), im, mk)[0].mean()
        row += [t, l]
    print(f"{d:14s} {full:7.4f} | " + " ".join(f"{v:7.4f}" for v in row))

print("\nfull = untruncated full FT | tr = rank-r truncation of full FT | lo = LoRA r")
print("tr close to full  -> full-FT solution IS effectively low-rank")
print("lo > tr           -> LoRA is optimised low-rank, not a static projection")

In [ ]:
import torch, torch.nn.functional as F, numpy as np

@torch.no_grad()
def block_features(model, loader, n_blocks=12):
    """Per-block token features (mean over patches) for CKA. Returns list of [N, D]."""
    feats = [[] for _ in range(n_blocks)]
    is_peft = hasattr(model.encoder, 'base_model')
    enc = model.encoder.base_model if is_peft else model.encoder
    for x, _ in loader:
        x = x.to(DEVICE)
        out = enc(pixel_values=x, output_hidden_states=True)
        hs = out.hidden_states  # tuple: embeddings + 12 blocks
        for b in range(n_blocks):
            h = hs[b + 1][:, 1:, :].mean(dim=1)   # skip CLS, mean over patches
            feats[b].append(h.cpu())
    return [torch.cat(f) for f in feats]

def linear_cka(X, Y):
    """Debiased linear CKA (Kornblith 2019)."""
    X = X - X.mean(0, keepdim=True); Y = Y - Y.mean(0, keepdim=True)
    XtX = (X.T @ X); YtY = (Y.T @ Y)
    hsic = (X @ Y.T).pow(2).sum()
    return (hsic / (XtX.norm() * YtY.norm() + 1e-12)).item()

# use one representative test set (pick in-domain + one OOD)
from torch.utils.data import DataLoader
def loader_for(d):
    im, mk = clean_sets[d]
    return DataLoader(PolypDataset(im, mk, augment=False), batch_size=16, shuffle=False, num_workers=2)

# load the three models (seed 42)
pre = DINOv2SegFull().to(DEVICE)  # pretrained encoder inside, decoder random — only encoder used
import copy
from transformers import AutoModel
pre.encoder.load_state_dict(AutoModel.from_pretrained('facebook/dinov2-small').state_dict())
pre.eval()

m_lora = DINOv2SegLoRAx(rank=32).to(DEVICE)
m_lora.load_state_dict(torch.load(f'{CKPT}/lora_r32_polyp_s42.pth')); m_lora.eval()

m_full = DINOv2SegFull().to(DEVICE)
m_full.load_state_dict(torch.load(f'{CKPT}/full_polyp_s42.pth')); m_full.eval()

for dset in ['Kvasir', 'ETIS']:
    ld = loader_for(dset)
    fp = block_features(pre,    ld)
    fl = block_features(m_lora, ld)
    ff = block_features(m_full, ld)
    print(f"\n=== {dset} ===")
    print(f"{'block':>5s} {'CKA(pre,LoRA)':>14s} {'CKA(pre,full)':>14s} {'CKA(LoRA,full)':>15s}")
    for b in range(12):
        c_pl = linear_cka(fp[b], fl[b])
        c_pf = linear_cka(fp[b], ff[b])
        c_lf = linear_cka(fl[b], ff[b])
        print(f"{b:5d} {c_pl:14.3f} {c_pf:14.3f} {c_lf:15.3f}")

In [ ]:
#check
!ls /content/drive/MyDrive/thesis_ckpts | grep polyp

In [ ]:
# ===== CKA — same code in both domains, debiased + plain side by side =====
import numpy as np, torch, json
from torch.utils.data import DataLoader
from peft import PeftModel

# ---- ONLY THIS BLOCK DIFFERS BETWEEN NOTEBOOKS ----
DOMAIN   = 'polyp'                    # in the skin notebook: 'skin'
DS_CLASS = PolypDataset                # in the skin notebook: ISICDataset
CKA_SETS = ['Kvasir', 'ETIS']          # in the skin notebook: ['PH2']
TAGS = {'frozen':'frozen_polyp_s42', 'lora_r8':'lora_r8_polyp_s42',
        'lora_r32':'lora_r32_polyp_s42', 'full':'full_polyp_s42'}
# skin: {'frozen':'frozen_aug','lora_r8':'lora_r8_s42',
#        'lora_r32':'lora_r32_s42','full':'full_s42'}
# ---------------------------------------------------

N_CKA, N_BLOCKS = 100, 12   # n fixed: the CKA value is sensitive to sample size

@torch.no_grad()
def block_feats(model, imgs, masks):
    model.eval()
    enc = model.encoder
    core = enc.base_model.model if isinstance(enc, PeftModel) else enc
    ld = DataLoader(DS_CLASS(imgs[:N_CKA], masks[:N_CKA], augment=False),
                    batch_size=8, shuffle=False, num_workers=2)
    acc = [[] for _ in range(N_BLOCKS)]
    for x, _ in ld:                                  # fp32, NO autocast
        hs = core(pixel_values=x.to(DEVICE), output_hidden_states=True).hidden_states
        for b in range(N_BLOCKS):
            acc[b].append(hs[b+1][:, 1:, :].float().mean(1).cpu())
    return [torch.cat(a).numpy().astype(np.float64) for a in acc]

def _dhsic(K, L):                                    # Nguyen et al. 2021
    n = K.shape[0]; K, L = K.copy(), L.copy()
    np.fill_diagonal(K, 0.0); np.fill_diagonal(L, 0.0)
    t  = (K @ L).trace()
    t -= (K.sum(0) @ L.sum(0)) * 2.0 / (n - 2)
    t += K.sum() * L.sum() / ((n - 1) * (n - 2))
    return t / (n * (n - 3))

def cka_debiased(X, Y):
    K, L = X @ X.T, Y @ Y.T
    return float(_dhsic(K, L) / np.sqrt(max(_dhsic(K,K),1e-12) * max(_dhsic(L,L),1e-12)))

def cka_plain(X, Y):                        # the earlier polyp computation — for comparison
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    num = np.linalg.norm(Y.T @ X, 'fro') ** 2
    den = np.linalg.norm(X.T @ X, 'fro') * np.linalg.norm(Y.T @ Y, 'fro')
    return float(num / (den + 1e-12))

def load(key):
    if key == 'frozen':   m = DINOv2SegFrozen().to(DEVICE)
    elif key == 'full':   m = DINOv2SegFull().to(DEVICE)
    else:                 m = DINOv2SegLoRAx(rank=int(key.split('r')[-1])).to(DEVICE)
    m.load_state_dict(torch.load(f'{CKPT}/{TAGS[key]}.pth')); m.eval(); return m

out = {}
for dset in CKA_SETS:
    imgs, masks = clean_sets[dset]
    m_pre = DINOv2SegFrozen().to(DEVICE); m_pre.eval()      # untrained encoder
    F = {'pre': block_feats(m_pre, imgs, masks)}
    del m_pre; torch.cuda.empty_cache()
    for k in TAGS:
        m = load(k); F[k] = block_feats(m, imgs, masks)
        del m; torch.cuda.empty_cache()

    print(f"\n=== {DOMAIN} / {dset} — CKA(pretrained, X), debiased ===")
    print(f"{'blk':>3s} " + " ".join(f'{k:>10s}' for k in TAGS) + f" {'L32-vs-full':>12s}")
    rows = {}
    for b in range(N_BLOCKS):
        r = {k: cka_debiased(F['pre'][b], F[k][b]) for k in TAGS}
        r['lora32_vs_full'] = cka_debiased(F['lora_r32'][b], F['full'][b])
        rows[b] = r
        print(f"{b:3d} " + " ".join(f"{r[k]:10.3f}" for k in TAGS)
              + f" {r['lora32_vs_full']:12.3f}")

    # did the choice of estimator actually matter?
    print(f"\n{'blk':>3s} {'lora32 debiased':>16s} {'lora32 plain':>14s} {'difference':>8s}")
    for b in range(N_BLOCKS):
        d = rows[b]['lora_r32']
        p = cka_plain(F['pre'][b], F['lora_r32'][b])
        print(f"{b:3d} {d:16.3f} {p:14.3f} {d-p:+8.3f}")

    out[dset] = rows

np.save(f'{CKPT}/cka_{DOMAIN}_fixed.npy', out, allow_pickle=True)
print(f"\nsaved: cka_{DOMAIN}_fixed.npy")

In [ ]:
# 40 ep rank sweep
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

TOTAL_EP, PATIENCE = 40, 8
sweep40 = {}

for r in [4, 8, 16, 32]:
    set_seed(42)
    print("="*50, f"\nLoRA r={r} | 40ep\n", "="*50)
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    train_or_load(m, opt, f'lora_r{r}_polyp_s42_e40',
                  total_ep=TOTAL_EP, warmup_ep=2, patience=PATIENCE)
    sweep40[r] = m

set_seed(42)
print("="*50, "\nFULL FT | 40ep\n", "="*50)
m_full40 = DINOv2SegFull().to(DEVICE)
opt = torch.optim.AdamW(build_llrd_params(m_full40, 5e-6, 1e-4, decay=0.75))
train_or_load(m_full40, opt, 'full_polyp_s42_e40',
              total_ep=TOTAL_EP, warmup_ep=3, patience=PATIENCE)

order = ['Kvasir','CVC-ClinicDB','CVC-ColonDB','ETIS','CVC-T']
rank40 = {}
print(f"\n=== 40ep ===\n{'rank':>5s} " + " ".join(f"{d:>13s}" for d in order))
for r in [4, 8, 16, 32]:
    row = {d: eval_vec(sweep40[r], *clean_sets[d])[0].mean() for d in order}
    rank40[r] = row
    print(f"{r:5d} " + " ".join(f"{row[d]:13.4f}" for d in order))
full40 = {d: eval_vec(m_full40, *clean_sets[d])[0].mean() for d in order}
print(f"{'full':>5s} " + " ".join(f"{full40[d]:13.4f}" for d in order))

old = np.load(f'{CKPT}/rank_sweep_polyp.npy', allow_pickle=True).item()
print(f"\n=== rank effect (r32 - r4): 20ep vs 40ep ===")
for d in order:
    o = old['rank_dsc'][d][32] - old['rank_dsc'][d][4]
    n = rank40[32][d] - rank40[4][d]
    print(f"{d:14s} 20ep {o:+.4f} | 40ep {n:+.4f} | change {n-o:+.4f}")

np.save(f'{CKPT}/rank_sweep_polyp_e40.npy',
        {'rank_dsc': rank40, 'full_dsc': full40}, allow_pickle=True)

U-net Start

In [ ]:
#cell A
# Stronger augmentation for from-scratch training (nnU-Net-style).
# Kept separate from the DINOv2 pipeline so the adaptation experiments are unchanged.
class SegDatasetStrong(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False):
        self.image_paths, self.mask_paths = image_paths, mask_paths
        aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.25, rotate_limit=30,
                               border_mode=0, p=0.5),
            A.ElasticTransform(alpha=120, sigma=8, border_mode=0, p=0.2),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.3),
            A.RandomGamma(gamma_limit=(70, 150), p=0.3),
            A.GaussNoise(p=0.2),
        ] if augment else []
        self.transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *aug,
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, i):
        image = np.array(Image.open(self.image_paths[i]).convert('RGB'))
        mask  = np.array(Image.open(self.mask_paths[i]).convert('L'))
        t = self.transform(image=image, mask=mask)
        return t['image'].float(), (t['mask'].float()/255.0 > 0.5).float().unsqueeze(0)

tr_ds, va_ds = train_loader.dataset, val_loader.dataset
train_loader_s = DataLoader(SegDatasetStrong(tr_ds.image_paths, tr_ds.mask_paths, augment=True),
                            batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader_s   = DataLoader(SegDatasetStrong(va_ds.image_paths, va_ds.mask_paths, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_loader_s.dataset)} | val {len(val_loader_s.dataset)}")

In [ ]:
# ResNet-style encoder U-Net trained from scratch, with deep supervision.
# nnU-Net Revisited (Isensee 2024) reports residual-encoder U-Nets as the strongest recipe.
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.n1 = nn.InstanceNorm2d(cout, affine=True)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.n2 = nn.InstanceNorm2d(cout, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)
        self.skip = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                   nn.InstanceNorm2d(cout, affine=True))
                     if (stride != 1 or cin != cout) else nn.Identity())
    def forward(self, x):
        r = self.skip(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        return self.act(x + r)

class ResUNet(nn.Module):
    def __init__(self, base=32, depth=5, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        ch = [base * 2**i for i in range(depth)]          # 32,64,128,256,512
        self.stem = ResBlock(3, ch[0])
        self.enc  = nn.ModuleList([ResBlock(ch[i], ch[i+1], stride=2) for i in range(depth-1)])
        self.up   = nn.ModuleList([nn.ConvTranspose2d(ch[i+1], ch[i], 2, 2) for i in reversed(range(depth-1))])
        self.dec  = nn.ModuleList([ResBlock(ch[i]*2, ch[i]) for i in reversed(range(depth-1))])
        self.head = nn.ModuleList([nn.Conv2d(ch[i], 1, 1) for i in reversed(range(depth-1))])
    def forward(self, x):
        feats = [self.stem(x)]
        for e in self.enc: feats.append(e(feats[-1]))
        d, outs = feats[-1], []
        for i, (u, dec, hd) in enumerate(zip(self.up, self.dec, self.head)):
            d = dec(torch.cat([u(d), feats[-2-i]], dim=1))
            outs.append(hd(d))
        full = F.interpolate(outs[-1], size=IMG_SIZE, mode='bilinear', align_corners=False)
        if not (self.training and self.deep_sup): return full
        return [full] + [F.interpolate(o, size=IMG_SIZE, mode='bilinear', align_corners=False)
                         for o in outs[:-1]]

def dice_bce_ds(out, target):
    """Deep supervision: weights halve per level, normalised."""
    if not isinstance(out, list): return dice_bce(out, target)
    w = [1 / 2**i for i in range(len(out))]
    w = [x / sum(w) for x in w]
    return sum(wi * dice_bce(o, target) for wi, o in zip(w, out))

m = ResUNet()
print(f"ResUNet trainable params: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
def make_poly(opt, total_ep, power=0.9):
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (1 - ep/total_ep)**power)

def train_scratch(m, opt, tag, total_ep=150, patience=30):
    sch = make_poly(opt, total_ep)
    scaler = torch.amp.GradScaler('cuda')
    best, bad = -1, 0
    for ep in range(total_ep):
        m.train()
        for x, y in train_loader_s:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = dice_bce_ds(m(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 12)
            scaler.step(opt); scaler.update()
        sch.step()
        vd, _ = metrics_per_image(m, val_loader_s)
        v = vd.mean(); flag = ''
        if v > best:
            best, bad = v, 0
            torch.save(m.state_dict(), f'{CKPT}/{tag}.pth'); flag = ' *'
        else:
            bad += 1
        if (ep+1) % 5 == 0 or flag:
            print(f"  ep {ep+1:3d}/{total_ep}  val DSC {v:.4f}{flag}", flush=True)
        if bad >= patience:
            print(f"  early stop (ep {ep+1})"); break
    m.load_state_dict(torch.load(f'{CKPT}/{tag}.pth'))
    return best

def train_or_load_scratch(m, opt, tag, **kw):
    p = f'{CKPT}/{tag}.pth'
    if os.path.exists(p):
        m.load_state_dict(torch.load(p))
        vd, _ = metrics_per_image(m, val_loader_s)
        print(f"  SKIP {tag} — val DSC {vd.mean():.4f}")
        return vd.mean()
    return train_scratch(m, opt, tag, **kw)

In [ ]:
# LR search on polyp. From-scratch training is LR-sensitive; a single default
# would be exactly the weak-baseline problem nnU-Net Revisited criticises.
scratch = {}
for lr in [1e-2, 3e-3, 1e-3]:
    set_seed(42)
    print("="*50, f"\nResUNet scratch | SGD lr={lr}\n", "="*50)
    m = ResUNet().to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.99,
                          nesterov=True, weight_decay=3e-5)
    b = train_or_load_scratch(m, opt, f'resunet_polyp_lr{lr}_s42', total_ep=150, patience=30)
    scratch[lr] = (m, b)

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>13s}" for d in order))
for lr, (m, b) in scratch.items():
    row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    print(f"{lr:8.0e} {b:8.4f} " + " ".join(f"{row[d]:13.4f}" for d in row))

In [ ]:
# Budget control: no run stopped early, so extend the best LR.
set_seed(42)
m = ResUNet().to(DEVICE)
opt = torch.optim.SGD(m.parameters(), lr=3e-3, momentum=0.99, nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m, opt, 'resunet_polyp_lr0.003_e300_s42', total_ep=300, patience=50)

row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
print(f"{'300ep':>8s} " + " ".join(f"{row[d]:13.4f}" for d in order))

Note: this cell overwrites train_loader and train_loader_s. Re-run cell 3 and cell A before training anything else in this notebook

In [ ]:
print(len(train_loader.dataset.image_paths))

In [ ]:
# less data
# Data-size sweep on polyp: shrink the TRAINING set only; val and test sets stay fixed.
# Isolates data scale from modality and difficulty.
# Epoch budgets scale inversely with subset size so every run receives comparable
# optimizer steps; early stopping terminates runs that converge sooner.
import random

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

_tr = train_loader.dataset
TR_IMGS, TR_MASKS = _tr.image_paths, _tr.mask_paths

FRACS   = [0.10, 0.25, 0.50]                      # 1.00 already trained for both models
EP_LORA = {10: 400, 25: 160, 50: 80,  100: 40}
EP_RES  = {10: 3000, 25: 1200, 50: 600, 100: 300}
order = ['Kvasir', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS', 'CVC-T']

def make_subset(frac, seed=42):
    """Identical subset for both models, so the comparison is paired."""
    idx = list(range(len(TR_IMGS)))
    random.Random(seed).shuffle(idx)
    idx = sorted(idx[:int(len(TR_IMGS) * frac)])
    return [TR_IMGS[i] for i in idx], [TR_MASKS[i] for i in idx]

results = {}

for f in FRACS:
    im, mk = make_subset(f)
    pct = int(f * 100)
    print("="*60, f"\n{pct}% of training set ({len(im)} images) | "
          f"LoRA {EP_LORA[pct]}ep, ResUNet {EP_RES[pct]}ep\n", "="*60)

    # --- DINOv2 + LoRA r=32 ---
    train_loader = DataLoader(PolypDataset(im, mk, augment=True), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    set_seed(42)
    m = DINOv2SegLoRAx(rank=32).to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=1e-4, weight_decay=1e-4)
    train_or_load(m, opt, f'lora_r32_polyp_f{pct}_s42',
                  total_ep=EP_LORA[pct], warmup_ep=2, patience=8)
    results[(pct, 'lora')] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    del m; torch.cuda.empty_cache()

    # --- ResUNet from scratch ---
    train_loader_s = DataLoader(SegDatasetStrong(im, mk, augment=True), batch_size=BATCH_SIZE,
                                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    set_seed(42)
    m = ResUNet().to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=3e-3, momentum=0.99,
                          nesterov=True, weight_decay=3e-5)
    train_or_load_scratch(m, opt, f'resunet_polyp_f{pct}_s42',
                          total_ep=EP_RES[pct], patience=50)
    results[(pct, 'res')] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    del m; torch.cuda.empty_cache()

np.save(f'{CKPT}/datasize_polyp.npy', results, allow_pickle=True)
print("\nsaved: datasize_polyp.npy")

In [ ]:
IND = ['Kvasir', 'CVC-ClinicDB']
OOD = ['CVC-ColonDB', 'ETIS', 'CVC-T']

# 100% figures from the runs already completed
results[(100, 'res')]  = {'Kvasir':0.861,'CVC-ClinicDB':0.883,
                          'CVC-ColonDB':0.724,'ETIS':0.618,'CVC-T':0.794}
results[(100, 'lora')] = {'Kvasir':0.888,'CVC-ClinicDB':0.868,
                          'CVC-ColonDB':0.769,'ETIS':0.698,'CVC-T':0.867}

print(f"{'%':>5s} {'n':>6s} | {'ResU in':>8s} {'DINO in':>8s} {'diff':>8s} "
      f"| {'ResU OOD':>9s} {'DINO OOD':>9s} {'diff':>8s}")
print('-'*74)
for pct in [10, 25, 50, 100]:
    if (pct,'res') not in results: continue
    r, l = results[(pct,'res')], results[(pct,'lora')]
    ri, li = np.mean([r[d] for d in IND]), np.mean([l[d] for d in IND])
    ro, lo = np.mean([r[d] for d in OOD]), np.mean([l[d] for d in OOD])
    print(f"{pct:5d} {int(1305*pct/100):6d} | {ri:8.4f} {li:8.4f} {ri-li:+8.4f} "
          f"| {ro:9.4f} {lo:9.4f} {ro-lo:+8.4f}")

# Cross-comparison: how little data does each model need to match the other at 100%?
print(f"\n--- vs DINOv2 @100% (in {np.mean([results[(100,'lora')][d] for d in IND]):.4f} "
      f"| OOD {np.mean([results[(100,'lora')][d] for d in OOD]):.4f}) ---")
ref_in  = np.mean([results[(100,'lora')][d] for d in IND])
ref_ood = np.mean([results[(100,'lora')][d] for d in OOD])
for pct in [10, 25, 50, 100]:
    if (pct,'res') not in results: continue
    r = results[(pct,'res')]
    ri, ro = np.mean([r[d] for d in IND]), np.mean([r[d] for d in OOD])
    print(f"ResUNet @{pct:3d}%: in {ri:.4f} ({ri-ref_in:+.4f}) | OOD {ro:.4f} ({ro-ref_ood:+.4f})")

print(f"\n--- vs ResUNet @100% (in {np.mean([results[(100,'res')][d] for d in IND]):.4f} "
      f"| OOD {np.mean([results[(100,'res')][d] for d in OOD]):.4f}) ---")
ref_in  = np.mean([results[(100,'res')][d] for d in IND])
ref_ood = np.mean([results[(100,'res')][d] for d in OOD])
for pct in [10, 25, 50, 100]:
    if (pct,'lora') not in results: continue
    l = results[(pct,'lora')]
    li, lo = np.mean([l[d] for d in IND]), np.mean([l[d] for d in OOD])
    print(f"DINOv2  @{pct:3d}%: in {li:.4f} ({li-ref_in:+.4f}) | OOD {lo:.4f} ({lo-ref_ood:+.4f})")

In [ ]:
import random

# Restore the full training list and the subset helper
_tr = train_loader.dataset
FULL_TR_IMGS, FULL_TR_MASKS = list(_tr.image_paths), list(_tr.mask_paths)
assert len(FULL_TR_IMGS) == 1305, f"expected 1305, got {len(FULL_TR_IMGS)} — run cell 3 first"

def make_subset(frac, seed=42):
    idx = list(range(len(FULL_TR_IMGS)))
    random.Random(seed).shuffle(idx)
    idx = sorted(idx[:int(len(FULL_TR_IMGS) * frac)])
    return [FULL_TR_IMGS[i] for i in idx], [FULL_TR_MASKS[i] for i in idx]

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

order = ['Kvasir', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS', 'CVC-T']
print(f"restored: {len(FULL_TR_IMGS)} training images")

In [ ]:
# Low-data configuration search for the from-scratch baseline at 25% (326 images).
# The LR was tuned on the full training set; this checks whether the from-scratch
# model is limited by its configuration rather than by the approach itself.
import copy

im25, mk25 = make_subset(0.25)
print(f"subset: {len(im25)} images")

# --- stronger augmentation variant ---
class SegDatasetAggressive(SegDatasetStrong):
    def __init__(self, image_paths, mask_paths, augment=False):
        Dataset.__init__(self)
        self.image_paths, self.mask_paths = image_paths, mask_paths
        aug = [
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.4, rotate_limit=45,
                               border_mode=0, p=0.8),
            A.ElasticTransform(alpha=200, sigma=10, border_mode=0, p=0.4),
            A.GridDistortion(p=0.3),
            A.RandomBrightnessContrast(0.4, 0.4, p=0.5),
            A.RandomGamma(gamma_limit=(60, 160), p=0.5),
            A.GaussNoise(p=0.3),
            A.CoarseDropout(p=0.3),
        ] if augment else []
        self.transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *aug,
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])

# --- ResUNet with decoder dropout ---
class ResUNetDrop(ResUNet):
    def __init__(self, base=32, depth=5, deep_sup=True, p_drop=0.2):
        super().__init__(base, depth, deep_sup)
        self.drop = nn.Dropout2d(p_drop)
    def forward(self, x):
        feats = [self.stem(x)]
        for e in self.enc: feats.append(e(feats[-1]))
        d, outs = self.drop(feats[-1]), []
        for i, (u, dec, hd) in enumerate(zip(self.up, self.dec, self.head)):
            d = dec(torch.cat([u(d), feats[-2-i]], dim=1))
            outs.append(hd(self.drop(d)))
        full = F.interpolate(outs[-1], size=IMG_SIZE, mode='bilinear', align_corners=False)
        if not (self.training and self.deep_sup): return full
        return [full] + [F.interpolate(o, size=IMG_SIZE, mode='bilinear', align_corners=False)
                         for o in outs[:-1]]

CONFIGS = {
    'baseline':   dict(model=lambda: ResUNet(base=32),        wd=3e-5, ds=SegDatasetStrong),
    'small':      dict(model=lambda: ResUNet(base=16),        wd=3e-5, ds=SegDatasetStrong),
    'heavy_wd':   dict(model=lambda: ResUNet(base=32),        wd=3e-4, ds=SegDatasetStrong),
    'dropout':    dict(model=lambda: ResUNetDrop(base=32),    wd=3e-5, ds=SegDatasetStrong),
    'aggr_aug':   dict(model=lambda: ResUNet(base=32),        wd=3e-5, ds=SegDatasetAggressive),
}

lowdata = {}
for name, cfg in CONFIGS.items():
    print("="*60, f"\n{name} @25%\n", "="*60)
    train_loader_s = DataLoader(cfg['ds'](im25, mk25, augment=True), batch_size=BATCH_SIZE,
                                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    set_seed(42)
    m = cfg['model']().to(DEVICE)
    n_par = sum(p.numel() for p in m.parameters())
    opt = torch.optim.SGD(m.parameters(), lr=3e-3, momentum=0.99,
                          nesterov=True, weight_decay=cfg['wd'])
    train_or_load_scratch(m, opt, f'resunet_polyp_f25_{name}_s42', total_ep=1200, patience=50)
    lowdata[name] = ({d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}, n_par)
    del m; torch.cuda.empty_cache()

# --- comparison ---
IND = ['Kvasir', 'CVC-ClinicDB']
OOD = ['CVC-ColonDB', 'ETIS', 'CVC-T']
print(f"\n{'config':12s} {'params':>10s} {'in-domain':>10s} {'OOD':>8s}")
print('-'*44)
for name, (row, n) in lowdata.items():
    ri = np.mean([row[d] for d in IND]); ro = np.mean([row[d] for d in OOD])
    print(f"{name:12s} {n:10,d} {ri:10.4f} {ro:8.4f}")
print(f"\nDINOv2 @25%: in-domain 0.8435 | OOD 0.7439")
print(f"DINOv2 @10%: in-domain 0.8046 | OOD 0.6761")

np.save(f'{CKPT}/lowdata_configs_polyp.npy', lowdata, allow_pickle=True)

BitFit

In [ ]:
# BitFit (Ben Zaken et al., ACL 2022): all encoder weights frozen, biases trained.
# It has no rank parameter, so it probes the capacity axis independently of LoRA.
class DINOv2SegBitFit(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('facebook/dinov2-small')
        for n, p in self.encoder.named_parameters():
            p.requires_grad = ('bias' in n)
        self.decoder = LinearDecoder(384)
    def forward(self, x):
        return self.decoder(self.encoder(pixel_values=x).last_hidden_state[:,1:,:])

_m = DINOv2SegBitFit()
enc_tr = sum(p.numel() for p in _m.encoder.parameters() if p.requires_grad)
dec_tr = sum(p.numel() for p in _m.decoder.parameters())
print(f"BitFit encoder trainable: {enc_tr:,}")
print(f"decoder: {dec_tr:,} | total: {enc_tr + dec_tr:,}")
print(f"reference — LoRA r=4: 74K | r=32: 590K | full FT: 22.1M")
del _m

In [ ]:
# BitFit trains ~0.1% of the encoder, so LoRA's 1e-4 would under-train it.
# Same optimizer structure as LoRA (single group); only the LR is searched.
# Selection is on in-domain validation DSC; OOD scores are logged, not used.
TOTAL_EP, WARMUP, PATIENCE = 40, 2, 8
LRS = [1e-2, 3e-3, 1e-3, 3e-4]

bitfit_val, bitfit_ood = {}, {}
for lr in LRS:
    set_seed(42)
    print("="*50, f"\nBitFit | lr={lr:.0e}\n", "="*50)
    m = DINOv2SegBitFit().to(DEVICE)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-4)
    bitfit_val[lr] = train_or_load(m, opt, f'bitfit_polyp_lr{lr:.0e}_s42',
                                   total_ep=TOTAL_EP, warmup_ep=WARMUP, patience=PATIENCE)
    bitfit_ood[lr] = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    del m; torch.cuda.empty_cache()

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>13s}" for d in order))
for lr in LRS:
    print(f"{lr:8.0e} {bitfit_val[lr]:8.4f} "
          + " ".join(f"{bitfit_ood[lr][d]:13.4f}" for d in order))

BEST_LR = max(bitfit_val, key=bitfit_val.get)
print(f"\nselected on val: {BEST_LR:.0e}  (val {bitfit_val[BEST_LR]:.4f})")

In [ ]:
# Capacity axis at a single budget: frozen -> BitFit -> LoRA r=4..32 -> full FT.
# All runs are seed 42, 40 epochs, identical warmup/patience and test sets.
# Trainable parameters are counted at runtime, not assumed.
set_seed(42)
m_fr40 = DINOv2SegFrozen().to(DEVICE)
opt = torch.optim.AdamW(m_fr40.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
train_or_load(m_fr40, opt, 'frozen_polyp_s42_e40', total_ep=40, warmup_ep=2, patience=8)

set_seed(42)
m_bf = DINOv2SegBitFit().to(DEVICE)
opt = torch.optim.AdamW([p for p in m_bf.parameters() if p.requires_grad],
                        lr=BEST_LR, weight_decay=1e-4)
train_or_load(m_bf, opt, f'bitfit_polyp_lr{BEST_LR:.0e}_s42',
              total_ep=40, warmup_ep=2, patience=8)

def n_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

cap = {}
cap['frozen'] = (n_trainable(m_fr40), {d: eval_vec(m_fr40, *clean_sets[d])[0] for d in order})
cap['BitFit'] = (n_trainable(m_bf),   {d: eval_vec(m_bf,   *clean_sets[d])[0] for d in order})

for r in [4, 8, 16, 32]:
    m = DINOv2SegLoRAx(rank=r).to(DEVICE)
    m.load_state_dict(torch.load(f'{CKPT}/lora_r{r}_polyp_s42_e40.pth'))
    cap[f'LoRA r={r}'] = (n_trainable(m), {d: eval_vec(m, *clean_sets[d])[0] for d in order})
    del m; torch.cuda.empty_cache()

m_f = DINOv2SegFull().to(DEVICE)
m_f.load_state_dict(torch.load(f'{CKPT}/full_polyp_s42_e40.pth'))
cap['full FT'] = (n_trainable(m_f), {d: eval_vec(m_f, *clean_sets[d])[0] for d in order})

print(f"\n{'method':>10s} {'trainable':>11s} " + " ".join(f"{d:>13s}" for d in order))
for name, (npar, row) in cap.items():
    print(f"{name:>10s} {npar:11,d} " + " ".join(f"{row[d].mean():13.4f}" for d in order))

np.save(f'{CKPT}/capacity_polyp_e40.npy',
        {k: {'params': v[0], **{d: v[1][d] for d in order}} for k, v in cap.items()},
        allow_pickle=True)

Plain Unet

In [ ]:
import random
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("set_seed ready")

In [ ]:
for n in ['clean_sets','eval_vec','metrics_per_image','train_loader_s','val_loader_s',
          'ResUNet','dice_bce_ds','train_scratch','train_or_load_scratch',
          'make_poly','set_seed','CKPT','IMG_SIZE']:
    print(f"{n:22s} {'OK' if n in globals() else 'MISSING'}")
print()
print('train_loader_s:', len(train_loader_s.dataset), '(1305 olmali)')
print('val_loader_s:  ', len(val_loader_s.dataset), '(145 olmali)')

In [ ]:
#A
# Plain U-Net: identical to ResUNet except the residual addition and the 1x1
# skip projection are removed. Channel widths, strided-conv downsampling,
# InstanceNorm, LeakyReLU and deep supervision are unchanged, so the residual
# connection is the only variable.
class PlainBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.n1 = nn.InstanceNorm2d(cout, affine=True)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.n2 = nn.InstanceNorm2d(cout, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        x = self.act(self.n1(self.c1(x)))
        return self.act(self.n2(self.c2(x)))

class PlainUNet(nn.Module):
    def __init__(self, base=32, depth=5, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        ch = [base * 2**i for i in range(depth)]          # 32,64,128,256,512
        self.stem = PlainBlock(3, ch[0])
        self.enc  = nn.ModuleList([PlainBlock(ch[i], ch[i+1], stride=2) for i in range(depth-1)])
        self.up   = nn.ModuleList([nn.ConvTranspose2d(ch[i+1], ch[i], 2, 2) for i in reversed(range(depth-1))])
        self.dec  = nn.ModuleList([PlainBlock(ch[i]*2, ch[i]) for i in reversed(range(depth-1))])
        self.head = nn.ModuleList([nn.Conv2d(ch[i], 1, 1) for i in reversed(range(depth-1))])
    def forward(self, x):
        feats = [self.stem(x)]
        for e in self.enc: feats.append(e(feats[-1]))
        d, outs = feats[-1], []
        for i, (u, dec, hd) in enumerate(zip(self.up, self.dec, self.head)):
            d = dec(torch.cat([u(d), feats[-2-i]], dim=1))
            outs.append(hd(d))
        full = F.interpolate(outs[-1], size=IMG_SIZE, mode='bilinear', align_corners=False)
        if not (self.training and self.deep_sup): return full
        return [full] + [F.interpolate(o, size=IMG_SIZE, mode='bilinear', align_corners=False)
                         for o in outs[:-1]]

m = PlainUNet()
print(f"PlainUNet trainable params: {sum(p.numel() for p in m.parameters()):,}   (ResUNet: 8.1M)")
del m

In [ ]:
#loader check
print(len(train_loader_s.dataset), len(val_loader_s.dataset))

In [ ]:
#B
# Same grid and budget as the ResUNet search (cell 46), so neither model is advantaged by tuning effort.
plain = {}
for lr in [1e-2, 3e-3, 1e-3]:
    set_seed(42)
    print("="*50, f"\nPlainUNet scratch | SGD lr={lr}\n", "="*50)
    m = PlainUNet().to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.99,
                          nesterov=True, weight_decay=3e-5)
    b = train_or_load_scratch(m, opt, f'plainunet_polyp_lr{lr}_s42', total_ep=150, patience=30)
    plain[lr] = (m, b)

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>13s}" for d in order))
for lr, (m, b) in plain.items():
    row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    print(f"{lr:8.0e} {b:8.4f} " + " ".join(f"{row[d]:13.4f}" for d in row))

In [ ]:
print(list(clean_sets.keys()))
order = ['Kvasir', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS', 'CVC-T']

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>13s}" for d in order))
for lr, (m, b) in plain.items():
    row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
    print(f"{lr:8.0e} {b:8.4f} " + " ".join(f"{row[d]:13.4f}" for d in row))

In [ ]:
set_seed(42)
print("="*50, "\nPlainUNet scratch | SGD lr=3e-4\n", "="*50)
m = PlainUNet().to(DEVICE)
opt = torch.optim.SGD(m.parameters(), lr=3e-4, momentum=0.99, nesterov=True, weight_decay=3e-5)
b = train_or_load_scratch(m, opt, 'plainunet_polyp_lr0.0003_s42', total_ep=150, patience=30)
plain[3e-4] = (m, b)

print(f"\n{'lr':>8s} {'val':>8s} " + " ".join(f"{d:>13s}" for d in order))
for lr in sorted(plain, reverse=True):
    mm, bb = plain[lr]
    row = {d: eval_vec(mm, *clean_sets[d])[0].mean() for d in order}
    print(f"{lr:8.0e} {bb:8.4f} " + " ".join(f"{row[d]:13.4f}" for d in row))

In [ ]:
BEST_LR = 1e-3

set_seed(42)
m = PlainUNet().to(DEVICE)
opt = torch.optim.SGD(m.parameters(), lr=BEST_LR, momentum=0.99, nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m, opt, 'plainunet_polyp_lr0.001_e300_s42', total_ep=300, patience=50)

row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
print(f"\n{'':>12s} " + " ".join(f"{d:>13s}" for d in order))
print(f"{'PlainUNet':>12s} " + " ".join(f"{row[d]:13.4f}" for d in order))

less data with unet

In [ ]:
for n in ['clean_sets','eval_vec','train_loader','train_or_load_scratch',
          'SegDatasetStrong','PlainUNet','set_seed','CKPT','BATCH_SIZE']:
    print(f"{n:22s} {'OK' if n in globals() else 'MISSING'}")
print('full train:', len(train_loader.dataset.image_paths))

In [ ]:
import random

# Rebuild the full training list; a subsetted loader would corrupt the split.
_tr = train_loader.dataset
assert len(_tr.image_paths) == 1305, f"train_loader is subsetted: {len(_tr.image_paths)}"
TR_IMGS, TR_MASKS = _tr.image_paths, _tr.mask_paths

def make_subset(frac, seed=42):
    idx = list(range(len(TR_IMGS)))
    random.Random(seed).shuffle(idx)
    idx = sorted(idx[:int(len(TR_IMGS) * frac)])
    return [TR_IMGS[i] for i in idx], [TR_MASKS[i] for i in idx]

order = ['Kvasir', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS', 'CVC-T']

# Same subset, same seed and same epoch budget as the ResUNet 25% run (cell 50),
# so the two from-scratch models are paired. LR is each model's own
# validation-selected value: 3e-3 for ResUNet, 1e-3 for PlainUNet.
im, mk = make_subset(0.25)
print("="*60, f"\npolyp 25% ({len(im)} images) | PlainUNet 1200ep\n", "="*60)

train_loader_s = DataLoader(SegDatasetStrong(im, mk, augment=True), batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
set_seed(42)
m = PlainUNet().to(DEVICE)
opt = torch.optim.SGD(m.parameters(), lr=1e-3, momentum=0.99,
                      nesterov=True, weight_decay=3e-5)
train_or_load_scratch(m, opt, 'plainunet_polyp_f25_s42', total_ep=1200, patience=50)

row = {d: eval_vec(m, *clean_sets[d])[0].mean() for d in order}
print(f"\n{'':12s} " + " ".join(f"{d:>13s}" for d in order))
print(f"{'PlainUNet':12s} " + " ".join(f"{row[d]:13.4f}" for d in row))

In [ ]:
d = np.load(f'{CKPT}/datasize_polyp.npy', allow_pickle=True).item()
print(d.keys())

In [ ]:
print(list(clean_sets.keys()))
print(CKPT)

In [ ]:
d0 = list(clean_sets)[0]
print("clean_sets key:", d0)
print("type:", type(clean_sets[d0]), "| len:", len(clean_sets[d0]))
for i, x in enumerate(clean_sets[d0]):
    print(f"  [{i}] {type(x).__name__}", end=" ")
    try: print(f"len={len(x)} first={x[0] if isinstance(x, list) else '...'}")
    except: print()

In [ ]:
import numpy as np, cv2, os

DOMAIN   = 'polyp'                              # 'polyp' | 'breast' | 'skin'
CAP_FILE = f'{CKPT}/capacity_polyp_e40.npy'     # breast: capacity_breast_e40.npy, skin: capacity_skin.npy
REF      = 'LoRA r=32'
OUT      = '/content/drive/MyDrive/thesis_ckpts/figexamples'
os.makedirs(OUT, exist_ok=True)

cap = np.load(CAP_FILE, allow_pickle=True).item()

for d in clean_sets:
    imgs, msks = clean_sets[d][0], clean_sets[d][1]
    dsc = np.asarray(cap[REF][d])
    assert len(dsc) == len(imgs), f"{d}: {len(dsc)} scores vs {len(imgs)} images"

    # median-scoring case: not the best, not the worst
    idx = int(np.argsort(dsc)[len(dsc) // 2])

    im = cv2.cvtColor(cv2.imread(imgs[idx]), cv2.COLOR_BGR2RGB)
    mk = cv2.imread(msks[idx], cv2.IMREAD_GRAYSCALE)
    mk = (mk > 127).astype(np.uint8)

    # keep the long side at 512 to bound file size
    s = 512 / max(im.shape[:2])
    if s < 1:
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)

    np.savez_compressed(f'{OUT}/{DOMAIN}__{d.replace(" ","_").replace("*","star")}.npz',
                        image=im, mask=mk, dsc=float(dsc[idx]), setname=d, domain=DOMAIN)
    print(f"{d:14s} median DSC {dsc[idx]:.3f}  (n={len(dsc)})  idx {idx}")

In [ ]:
import numpy as np, matplotlib.pyplot as plt, glob, os
from matplotlib.patches import Rectangle

OUT = '/content/drive/MyDrive/thesis_ckpts/figexamples'

# row order: (domain, [column labels in order])  — first column is the training source
LAYOUT = [
    ('polyp',  ['Kvasir', 'CVC-ColonDB', 'ETIS', 'CVC-T']),
    ('breast', ['BUSI-val', 'BrEaST', 'BUS-UCLM']),
    ('skin',   ['ISIC 2016*', 'ISIC 2017*', 'PH2']),
]

def load(dom, name):
    f = f'{OUT}/{dom}__{name.replace(" ","_").replace("*","star")}.npz'
    return np.load(f, allow_pickle=True) if os.path.exists(f) else None

ncol = max(len(r[1]) for r in LAYOUT)
fig, axes = plt.subplots(len(LAYOUT), ncol, figsize=(3.1 * ncol, 3.4 * len(LAYOUT)))

for r, (dom, names) in enumerate(LAYOUT):
    for c in range(ncol):
        ax = axes[r, c]; ax.axis('off')
        if c >= len(names):
            continue
        z = load(dom, names[c])
        if z is None:
            ax.text(.5, .5, f'missing\n{names[c]}', ha='center', va='center', fontsize=8)
            continue
        ax.imshow(z['image'])
        ax.contour(z['mask'], levels=[0.5], colors=['#ffd400'], linewidths=2.0)
        train = (c == 0)
        ax.set_title(f"{names[c]}{'  (train source)' if train else ''}\nDSC {float(z['dsc']):.3f}",
                     fontsize=9, fontweight='bold' if train else 'normal')
        if train:
            ax.add_patch(Rectangle((0, 0), z['image'].shape[1] - 1, z['image'].shape[0] - 1,
                                   fill=False, ec='#2b6cb0', lw=3))
    axes[r, 0].set_ylabel(dom, fontsize=11)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/thesis_ckpts/fig_examples.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
!ls /content/drive/MyDrive/thesis_ckpts/figexamples

In [ ]:
import numpy as np, cv2, os

DOMAIN = 'polyp'          # 'polyp' | 'breast' | 'skin'
OUT    = f'{CKPT}/figexamples'
os.makedirs(OUT, exist_ok=True)

# Midpoint of each set's file ordering, so the frame is not chosen on appearance.
# Override only where that frame carries an acquisition artefact.
IDX_OVERRIDE = {
    'BUS-UCLM': None,     # <-- i'll put an integer here after picking a cleaner frame
}

for d in clean_sets:
    imgs, msks = clean_sets[d][0], clean_sets[d][1]
    idx = IDX_OVERRIDE.get(d) or len(imgs) // 2

    im = cv2.cvtColor(cv2.imread(imgs[idx]), cv2.COLOR_BGR2RGB)
    mk = (cv2.imread(msks[idx], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    s = 512 / max(im.shape[:2])
    if s < 1:
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)

    np.savez_compressed(
        f'{OUT}/{DOMAIN}__{d.replace(" ","_").replace("*","star")}.npz',
        image=im, mask=mk, setname=d, domain=DOMAIN, index=idx, n=len(imgs))
    print(f"{d:20s} idx {idx}  n={len(imgs)}")